# Part 15: Blob Storage and Content Addressing

This notebook builds blob storage with content addressing. Blobs are binary data addressed by CID
and stored outside the database. Same data always produces the same CID, enabling deduplication.

**What you'll learn:**

- Content-addressed blob storage with CID deduplication
- Blob metadata tracking (mimeType, size)
- How CID-based addressing enables efficient storage

**Estimated Time:** 12-15 minutes

## Step 1: CID Helper

Blobs are addressed by CID. Same data always produces the same CID — this is the foundation of
deduplication.

In [ ]:
@interface CIDHelper : NSObject
- (NSString *)cidForData:(NSData *)data;
@end

@implementation CIDHelper
- (NSString *)cidForData:(NSData *)data {
    // Simplified hash: XOR-fold bytes into 4 hex bytes
    // Real implementations uses SHA-256 → base32 (CIDv1)
    int a = 0, b = 0, c = 0, d = 0;
    const char *bytes = [data bytes];
    int len = [data length];
    for (int i = 0; i < len; i++) {
        a = a ^ bytes[i];
        b = b ^ (bytes[i] << 1);
        c = c ^ (bytes[i] << 2);
        d = d ^ (bytes[i] << 3);
    }
    return [NSString stringWithFormat:@"bafyrei%02x%02x%02x%02x", a & 0xFF, b & 0xFF, c & 0xFF, d & 0xFF];
}
@end

## Step 2: Blob Store

`Blob services upload blobs, computes CIDs, and stores metadata. Storage layers handle the actual
file I/O. Here we use in-memory storage.

In [ ]:
@interface BlobStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *blobs;
@property (nonatomic, strong) NSMutableDictionary *metadata;
@property (nonatomic, strong) CIDHelper *cidHelper;
@property (nonatomic, assign) int referenceCount;
- (NSDictionary *)uploadBlob:(NSData *)data mimeType:(NSString *)mimeType forDid:(NSString *)did;
- (NSData *)getBlob:(NSString *)cid;
- (BOOL)deleteBlob:(NSString *)cid;
@end

@implementation BlobStore
- (instancetype)init {
    self = [super init];
    if (self) {
        _blobs = [NSMutableDictionary dictionary];
        _metadata = [NSMutableDictionary dictionary];
        _cidHelper = [[CIDHelper alloc] init];
    }
    return self;
}
- (NSDictionary *)uploadBlob:(NSData *)data mimeType:(NSString *)mimeType forDid:(NSString *)did {
    NSString *cid = [_cidHelper cidForData:data];
    // Check if blob already exists (deduplication)
    if ([_blobs objectForKey:cid] != nil) {
        NSLog(@"Blob already exists: %@ (dedup)", cid);
        NSMutableDictionary *meta = [NSMutableDictionary dictionaryWithDictionary:[_metadata objectForKey:cid]];
        int refs = [[meta objectForKey:@"refs"] intValue] + 1;
        [meta setObject:[NSString stringWithFormat:@"%d", refs] forKey:@"refs"];
        [_metadata setObject:meta forKey:cid];
        return meta;
    }
    // Store blob and metadata
    [_blobs setObject:data forKey:cid];
    NSDictionary *meta = @{
        @"cid": cid,
        @"mimeType": mimeType,
        @"size": [NSString stringWithFormat:@"%d", [data length]],
        @"did": did,
        @"refs": @"1"
    };
    [_metadata setObject:meta forKey:cid];
    NSLog(@"Uploaded blob: %@ (%d bytes, %@)", cid, [data length], mimeType);
    return meta;
}
- (NSData *)getBlob:(NSString *)cid {
    return [_blobs objectForKey:cid];
}
- (BOOL)deleteBlob:(NSString *)cid {
    if ([_blobs objectForKey:cid] == nil) return NO;
    [_blobs removeObjectForKey:cid];
    [_metadata removeObjectForKey:cid];
    NSLog(@"Deleted blob: %@", cid);
    return YES;
}
@end

## Step 3: Blob Deduplication

CID-based addressing means identical data produces identical CIDs. Two users uploading the same
image stores the bytes once but tracks two references.

In [ ]:
BlobStore *store = [[BlobStore alloc] init];

// Upload the same image data twice
NSData *imageData = [NSData dataWithBytes:"\x89PNG\x0d\x0a" length:6];

NSDictionary *meta1 = [store uploadBlob:imageData mimeType:@"image/png" forDid:@"did:plc:alice"];
NSDictionary *meta2 = [store uploadBlob:imageData mimeType:@"image/png" forDid:@"did:plc:bob"];

// Same CID for both uploads
NSLog(@"CID 1: %@", [meta1 objectForKey:@"cid"]);
NSLog(@"CID 2: %@", [meta2 objectForKey:@"cid"]);
NSLog(@"Same CID: %d", [[meta1 objectForKey:@"cid"] isEqualToString:[meta2 objectForKey:@"cid"]]);

// Reference count is 2
NSLog(@"References: %@", [meta2 objectForKey:@"refs"]);

// Different data produces different CID
NSData *otherData = [NSData dataWithBytes:"\x89JPG" length:4];
NSDictionary *meta3 = [store uploadBlob:otherData mimeType:@"image/jpeg" forDid:@"did:plc:alice"];
NSLog(@"Different CID: %d", ![[meta1 objectForKey:@"cid"] isEqualToString:[meta3 objectForKey:@"cid"]]);

// Retrieve blob
NSData *retrieved = [store getBlob:[meta1 objectForKey:@"cid"]];
NSLog(@"Retrieved %d bytes", [retrieved length]);

## Next Steps

You have learned content-addressed blob storage. Continue with schema migrations to understand how
databases evolve safely.

## What to Read Next

You now understand blob storage. Next:

- **Part 16: Firehose and Sync** — event streams and repository diffs
- **Part 17: Migrations** — schema evolution